# Qwen

**Qwen** (通义千问, "Tongyi Qianwen") is Alibaba's family of open-weight LLMs — one of the most complete open model lineups available. It spans tiny **0.6B** models that run on a phone up to **480B** Mixture-of-Experts giants, plus specialized branches: **Qwen-Coder** (code), **Qwen-VL** (vision), **Qwen-Audio**, and **Qwen-Math**. Most weights are **Apache-2.0** (commercial-friendly), downloadable from Hugging Face / ModelScope, and the hosted **DashScope** API is **OpenAI-compatible**.

**Domain:** Proprietary Models & Coding AI  ·  **recommended addition**  ·  **runnable:** yes  ·  _live cell gates on `os.getenv("DASHSCOPE_API_KEY")`_

## 1. What & Why

Qwen is Alibaba's answer to "we need open weights we can actually own, across every size and modality." Two things make it stand out:

- **Breadth.** Almost no other open family covers the whole range in one consistent design: dense models at **0.6B / 1.7B / 4B / 8B / 14B / 32B**, MoE models (**Qwen3-30B-A3B**, **Qwen3-235B-A22B**), and specialist forks for **code**, **vision**, **audio**, and **math**. You can prototype on a 4B model and scale to 235B without changing your prompt format or tokenizer.
- **Openness + license.** Most checkpoints are **Apache-2.0** — use them commercially, fine-tune them, ship them on-prem. The weights live on Hugging Face (`Qwen/...`) and ModelScope, and run through `transformers`, **vLLM**, **Ollama**, **llama.cpp**, and **SGLang**.

On top of that, **Qwen3** (2025) introduced **hybrid thinking**: a single model can answer instantly *or* deliberate in a `<think>…</think>` block, toggled per-request — no need to host two different models.

**The problem it solves.** You want strong, multilingual, tool-capable models but (a) need open weights for privacy/on-prem/fine-tuning, (b) want one consistent family from edge to datacenter, and/or (c) want frontier-ish coding (Qwen-Coder) without a closed vendor. Qwen covers all three.

**When to reach for it**

- **Self-hosting / fine-tuning** across a range of hardware (laptop → multi-GPU).
- **Coding** assistants and fill-in-the-middle completion (Qwen2.5/3-Coder).
- **Multilingual** workloads — Qwen is especially strong on Chinese and 100+ languages.
- **One model, two speeds** — instant answers *and* on-demand reasoning via the thinking toggle.

**When *not* to.** If you want the absolute top of a Western leaderboard, the deepest agentic-tooling ecosystem, or you have governance rules against China-origin model provenance even when self-hosted, a Western frontier API (Claude, GPT, Gemini) may fit better. For a hosted *reasoning-per-dollar* play you'd also weigh DeepSeek-R1 (see the `deepseek` notebook).

## 2. Mental Model

Think of Qwen as **"one tokenizer and one chat format, stamped out across every size, modality, and a thinking switch."** Learn the format once and it works from the 0.6B phone model to the 235B MoE.

```
                         ┌──────────────── the Qwen family ────────────────┐
   one ChatML format ───▶│  dense:  0.6B 1.7B 4B 8B 14B 32B                │
   one tokenizer         │  MoE:    30B-A3B   235B-A22B  (total-A-active)  │
   (Apache-2.0 weights)  │  forks:  Coder · VL (vision) · Audio · Math     │
                         └─────────────────────────┬───────────────────────┘
                                                   │  Qwen3 hybrid thinking
                                                   ▼
   prompt ─▶  enable_thinking=True  ─▶  <think> deliberation… </think> answer
             enable_thinking=False ─▶  answer immediately (no think block)

   ChatML wire format (what the tokenizer's chat template emits):
     <|im_start|>system\n{system}<|im_end|>
     <|im_start|>user\n{user}<|im_end|>
     <|im_start|>assistant\n            ← generation starts here
```

Three things to internalize:

1. **ChatML is the contract.** Qwen uses `<|im_start|>role … <|im_end|>` turn markers. `tokenizer.apply_chat_template(...)` builds this for you — never hand-concatenate strings, or the model's special-token boundaries break.
2. **MoE names encode the cost.** `Qwen3-235B-A22B` = **235B total parameters, ~22B Active per token**. You rent a huge brain but pay to run a small one each step (same idea as DeepSeek's MoE — see `mixture-of-experts`).
3. **Thinking is a switch, not a separate model.** Qwen3 toggles `enable_thinking` per call. On → it emits a `<think>…</think>` chain then the answer; off → straight to the answer. Same weights, two latency/quality profiles.

## 3. Key Concepts

- **Qwen3** — the current generation (2025). Dense **0.6B–32B** and MoE (**30B-A3B**, **235B-A22B**), Apache-2.0. Headline feature: **hybrid thinking** (`enable_thinking`).
- **Qwen2.5** — the prior, very widely deployed generation; still the default for many coder/VL/math forks. Dense **0.5B–72B**, 128K context (via YaRN), Apache-2.0 (except 3B/72B under a Qwen license).
- **Mixture-of-Experts (MoE).** `Qwen3-30B-A3B` = 30B total / ~3B active per token; `235B-A22B` = 235B / ~22B active. Big-model quality at small-model per-token compute. (See `mixture-of-experts`.)
- **Hybrid thinking / `enable_thinking`.** A per-request flag (or the `/think` and `/no_think` soft switches in the prompt). On → `<think>…</think>` reasoning before the answer; off → immediate answer. The thinking model is also published separately as `*-Thinking` variants.
- **ChatML + `apply_chat_template`.** Qwen's prompt format uses `<|im_start|>` / `<|im_end|>` role markers. Always build prompts with the tokenizer's chat template; it inserts the right special tokens and the trailing assistant cue.
- **Qwen2.5/3-Coder.** Code-specialized models with **fill-in-the-middle (FIM)** tokens (`<|fim_prefix|>`, `<|fim_suffix|>`, `<|fim_middle|>`), repo-level context, and strong tool/agent support. `Qwen3-Coder-480B-A35B` is the flagship coding MoE.
- **Qwen-VL / Qwen-Audio.** Multimodal branches: image+text (VL) and speech/audio (Audio) using the same chat backbone.
- **DashScope (Model Studio).** Alibaba Cloud's hosted API. An **OpenAI-compatible** endpoint at `https://dashscope-intl.aliyuncs.com/compatible-mode/v1` (or `dashscope.aliyuncs.com` in China) — a `base_url` swap. Model ids like `qwen-plus`, `qwen-max`, `qwen3-coder-plus`.
- **Function calling / Qwen-Agent.** Native tool-calling support plus the `qwen-agent` framework for ReAct-style agents, code interpreter, and RAG.
- **Long context + YaRN.** Native 32K, extendable to 128K+ via YaRN rope-scaling (a config flag, not retraining).

## 4. Setup

Three ways to use Qwen — pick by whether you want *local weights*, *a hosted API*, or *the easy local path*.

```bash
# A) Local weights with Hugging Face transformers (CPU works for the tiny models):
pip install "transformers>=4.51" torch accelerate
#   from transformers import AutoModelForCausalLM, AutoTokenizer
#   tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
#   model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")

# B) Hosted DashScope API (OpenAI-compatible — get a key in Alibaba Cloud Model Studio):
#    https://bailian.console.alibabacloud.com/  ->  API-KEY
export DASHSCOPE_API_KEY="sk-..."
pip install openai           # OpenAI SDK; just point base_url at DashScope

# C) Easiest local path — Ollama (quantized GGUF, one command):
ollama run qwen3:8b
ollama run qwen3:8b "/no_think  What is 2+2?"     # soft-switch off thinking
```

Minimal hosted call (OpenAI SDK, `base_url` swap):

```python
from openai import OpenAI
client = OpenAI(api_key=os.environ["DASHSCOPE_API_KEY"],
                base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1")
resp = client.chat.completions.create(
    model="qwen-plus",
    messages=[{"role": "user", "content": "Reverse the string 'qwen'."}],
    extra_body={"enable_thinking": False},   # Qwen3 thinking toggle
)
print(resp.choices[0].message.content)
```

The cells below run top-to-bottom in a **fresh kernel with no extra installs and no API key**: the ChatML and thinking-mode examples are pure Python, and the live API call is gated behind `os.getenv("DASHSCOPE_API_KEY")`.

In [1]:
# This notebook runs with or without an API key or heavy ML deps.
# To run the live example:  export DASHSCOPE_API_KEY="sk-..."   (optionally: pip install openai)
import os

api_key = os.getenv("DASHSCOPE_API_KEY")

try:
    import transformers  # noqa: F401
    have_transformers = True
except ImportError:
    have_transformers = False

print("DASHSCOPE_API_KEY :", "set" if api_key else "(unset -- live API calls skipped)")
print("transformers      :", "installed" if have_transformers else "(not installed -- using pure-Python demos)")
print()
print("Hosted models : qwen-plus | qwen-max | qwen-turbo | qwen3-coder-plus (DashScope, OpenAI-compatible)")
print("Endpoint      : https://dashscope-intl.aliyuncs.com/compatible-mode/v1")
print("Open weights  : huggingface.co/Qwen  (Apache-2.0; dense 0.6B-32B, MoE 30B-A3B / 235B-A22B, +Coder/VL/Audio)")

/Users/danieldekerlegand/Development/ai-tutor/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DASHSCOPE_API_KEY : (unset -- live API calls skipped)
transformers      : installed

Hosted models : qwen-plus | qwen-max | qwen-turbo | qwen3-coder-plus (DashScope, OpenAI-compatible)
Endpoint      : https://dashscope-intl.aliyuncs.com/compatible-mode/v1
Open weights  : huggingface.co/Qwen  (Apache-2.0; dense 0.6B-32B, MoE 30B-A3B / 235B-A22B, +Coder/VL/Audio)


## 5. Worked Examples

### Example 1 — Build a Qwen ChatML prompt by hand (no network)

Every Qwen model speaks **ChatML**: turns wrapped in `<|im_start|>role … <|im_end|>`, ending with an open `<|im_start|>assistant` cue that tells the model "your turn." In real code `tokenizer.apply_chat_template()` does this. Reconstructing it by hand cements *why* you must never hand-concatenate raw strings — the special-token boundaries are the contract.

In [2]:
# Reproduce exactly what tokenizer.apply_chat_template(..., add_generation_prompt=True)
# emits for a Qwen3 chat model. (No model download -- this is the wire format.)
IM_START, IM_END = "<|im_start|>", "<|im_end|>"

def build_qwen_prompt(messages, add_generation_prompt=True):
    parts = []
    for m in messages:
        parts.append(f"{IM_START}{m['role']}\n{m['content']}{IM_END}")
    text = "\n".join(parts)
    if add_generation_prompt:
        text += f"\n{IM_START}assistant\n"   # open turn -> model generates here
    return text

messages = [
    {"role": "system", "content": "You are a terse assistant."},
    {"role": "user",   "content": "Name the capital of France."},
]

prompt = build_qwen_prompt(messages)
print(prompt)
print("\n--- the model continues after the last line, then emits", repr(IM_END), "to stop ---")

# Why this matters: <|im_end|> is the EOS-equivalent that ends an assistant turn.
# Hand-built strings that omit it (or fuse roles) corrupt generation and stop tokens.
assert prompt.endswith(f"{IM_START}assistant\n")
assert prompt.count(IM_START) == 3 and prompt.count(IM_END) == 2  # 2 closed turns + 1 open

<|im_start|>system
You are a terse assistant.<|im_end|>
<|im_start|>user
Name the capital of France.<|im_end|>
<|im_start|>assistant


--- the model continues after the last line, then emits '<|im_end|>' to stop ---


### Example 2 — Qwen3 hybrid thinking: parse `<think>` and toggle it (no network)

Qwen3's signature feature is the **thinking switch**. With `enable_thinking=True` (or a `/think` hint), the model emits a `<think>…</think>` deliberation block *before* the answer; with it off (`/no_think`), it answers immediately. Your code has to **split the think block from the answer** — and decide what to keep in history.

In [3]:
import re

# A representative Qwen3 generation WITH thinking enabled (what you'd get back as raw text):
raw_thinking = (
    "<think>\n"
    "The user wants 17 * 24. 17 * 24 = 17 * 25 - 17 = 425 - 17 = 408.\n"
    "</think>\n"
    "17 * 24 = 408."
)

# ...and the SAME query with thinking disabled (/no_think): straight to the answer.
raw_no_think = "17 * 24 = 408."

def split_thinking(text):
    """Return (reasoning, answer). Mirrors how you'd post-process Qwen3 output."""
    m = re.search(r"<think>(.*?)</think>", text, flags=re.DOTALL)
    reasoning = m.group(1).strip() if m else ""
    answer = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    return reasoning, answer

for label, raw in [("enable_thinking=True", raw_thinking), ("/no_think", raw_no_think)]:
    reasoning, answer = split_thinking(raw)
    print(f"[{label}]")
    print("  reasoning:", repr(reasoning) if reasoning else "(none)")
    print("  answer   :", repr(answer))
    print()

# Rule of thumb (same as other reasoning models): show or log the <think> block,
# but carry only the ANSWER into multi-turn history -- don't feed reasoning back.
_, ans = split_thinking(raw_thinking)
history = [
    {"role": "user", "content": "What is 17 * 24?"},
    {"role": "assistant", "content": ans},        # answer only, think block dropped
]
assert "<think>" not in history[-1]["content"]
print("Next-turn history keeps the answer only:", history[-1])

[enable_thinking=True]
  reasoning: 'The user wants 17 * 24. 17 * 24 = 17 * 25 - 17 = 425 - 17 = 408.'
  answer   : '17 * 24 = 408.'

[/no_think]
  reasoning: (none)
  answer   : '17 * 24 = 408.'

Next-turn history keeps the answer only: {'role': 'assistant', 'content': '17 * 24 = 408.'}


### Example 3 — Call the live Qwen API via the OpenAI-compatible endpoint (gated on `DASHSCOPE_API_KEY`)

The real thing. With `DASHSCOPE_API_KEY` set this sends a live request to DashScope; otherwise it prints the exact call shape so the notebook still executes cleanly. We use plain `urllib` (no SDK required) — the body is **OpenAI-compatible**, so swapping in the `openai` client is a one-liner (`base_url=` the endpoint below).

In [4]:
# Live call -- gated so the notebook runs with or without a key.
import os, json, urllib.request, urllib.error

ENDPOINT = "https://dashscope-intl.aliyuncs.com/compatible-mode/v1/chat/completions"

def qwen_ask(question, model="qwen-plus", thinking=False):
    body = json.dumps({
        "model": model,
        "messages": [{"role": "user", "content": question}],
        "max_tokens": 256,
        "enable_thinking": thinking,   # Qwen3 hybrid-thinking toggle (DashScope honors it)
    }).encode()
    req = urllib.request.Request(
        ENDPOINT, data=body,
        headers={"Authorization": f"Bearer {os.environ['DASHSCOPE_API_KEY']}",
                 "Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)

if api_key:
    try:
        data = qwen_ask("Reverse the string 'qwen' and explain in one sentence.")
        msg = data["choices"][0]["message"]
        print("ANSWER:", msg["content"].strip())
        u = data.get("usage", {})
        print("usage :", {k: u[k] for k in u if "token" in k})
    except urllib.error.HTTPError as e:
        print("Live call failed:", e.code, e.read().decode()[:200])
    except Exception as e:
        print("Live call failed:", type(e).__name__, e)
else:
    print("Skipping live API call (set DASHSCOPE_API_KEY to run).")
    print("Request body would be:")
    print("  POST https://dashscope-intl.aliyuncs.com/compatible-mode/v1/chat/completions")
    print('  {"model": "qwen-plus", "messages": [{"role":"user","content":"..."}],')
    print('   "enable_thinking": false}')
    print("  -> choices[0].message.content   (the answer; <think> block if thinking on)")

Skipping live API call (set DASHSCOPE_API_KEY to run).
Request body would be:
  POST https://dashscope-intl.aliyuncs.com/compatible-mode/v1/chat/completions
  {"model": "qwen-plus", "messages": [{"role":"user","content":"..."}],
   "enable_thinking": false}
  -> choices[0].message.content   (the answer; <think> block if thinking on)


## 6. Gotchas & Pitfalls

- **Always use `apply_chat_template`.** Hand-building the `<|im_start|>/<|im_end|>` ChatML by concatenation drops or duplicates special tokens and breaks the stop boundary. Let the tokenizer build the prompt (Example 1 shows the format only so you recognize it).
- **`enable_thinking` lives in `extra_body` on the OpenAI SDK.** It's not a top-level OpenAI param, so `client.chat.completions.create(..., extra_body={"enable_thinking": False})`. With raw HTTP it's a normal top-level field. The `/think` and `/no_think` soft switches in the user message also work.
- **You pay for `<think>` tokens.** With thinking on, the deliberation counts as output tokens and latency. Turn it **off** for simple/format-bound tasks (extraction, classification) and on only when reasoning helps.
- **Strip `<think>` before parsing structured output.** If you ask for JSON with thinking on, the raw text is `<think>…</think>{json}`. Remove the think block first (Example 2) or you'll fail to parse.
- **`Qwen3-Coder` models are non-thinking by design.** The flagship coder doesn't emit `<think>`; don't wait for a reasoning block that never comes. Conversely, base (non-Instruct) checkpoints have **no** chat template at all — use the `-Instruct` / chat variants for conversation.
- **128K context needs YaRN enabled.** Qwen models are natively 32K; the longer windows require turning on YaRN rope-scaling in the model config. It's off by default, so long-context calls silently truncate if you forget.
- **Two DashScope regions.** `dashscope-intl.aliyuncs.com` (international) vs `dashscope.aliyuncs.com` (mainland China) — keys and model availability can differ. Pointing at the wrong base URL gives auth/404 errors.
- **License isn't uniformly Apache-2.0.** Most Qwen2.5/3 weights are Apache-2.0, but a few sizes (e.g. Qwen2.5-3B and -72B) ship under a separate Qwen license. Check the model card before commercial use.
- **Data residency / governance.** The hosted DashScope API runs on Alibaba Cloud; prompts leave your jurisdiction. Self-hosting the open weights sidesteps the API but not the model's provenance — the same consideration as DeepSeek.

## 7. When to Use vs Alternatives

| You need… | Reach for | Why |
|---|---|---|
| **Open weights across many sizes (edge → datacenter)** | **Qwen3 dense 0.6B–32B** | One format/tokenizer from phone-sized to server-sized; Apache-2.0. |
| **Cheap big-model quality, self-hosted** | **Qwen3-MoE (30B-A3B / 235B-A22B)** | MoE = big knowledge, small per-token compute; runs on fewer GPUs than dense equivalents. |
| **Open coding model / FIM completion** | **Qwen2.5 / 3-Coder** | Code-specialized, fill-in-the-middle, repo context, strong tool use. |
| **Multilingual (esp. Chinese)** | **Qwen (any)** | Trained heavily on Chinese + 100+ languages; class-leading there. |
| **One model, instant *and* reasoning** | **Qwen3 + thinking toggle** | Switch deliberation on/off per request instead of hosting two models. |
| **Vision / audio, open weights** | **Qwen-VL / Qwen-Audio** | Same backbone, multimodal input. |
| **Cheap hosted *reasoning*** | **DeepSeek-R1** (`deepseek` notebook) | If you don't need open weights, R1 is a strong hosted reasoning-per-dollar play. |
| **Top Western leaderboard, richest agent tooling, multimodal polish** | **Claude / GPT / Gemini** | Closed frontier APIs lead on agentic ecosystem and enterprise governance. |

**Honest trade-offs**

- **vs DeepSeek** — both are top-tier open Chinese families. Qwen wins on **breadth** (many sizes + Coder/VL/Audio/Math, easy to self-host) and the **thinking toggle**; DeepSeek's full V3/R1 push higher on raw reasoning but are 671B MoE and far heavier to run. Ironically, DeepSeek-R1's reasoning is most *usable* locally through **Qwen** distill backbones.
- **vs Llama / Mistral (other open families)** — Qwen generally matches or beats them on benchmarks and offers a wider size range and stronger multilingual/coding coverage; Llama/Mistral have larger Western ecosystems and tooling familiarity.
- **vs Claude / GPT / Gemini (closed frontier)** — Qwen wins on **openness, price, and self-hosting**; the closed labs lead on top-end frontier quality, agentic tool-use maturity, multimodal breadth, and Western data-governance comfort.

## 8. Resources

- **Qwen docs (Qwen3, usage, deployment)** — https://qwen.readthedocs.io/
- **Official blog (release notes, benchmarks)** — https://qwenlm.github.io/blog/
- **Open weights on Hugging Face** — https://huggingface.co/Qwen
- **GitHub (Qwen3 + reports)** — https://github.com/QwenLM/Qwen3
- **DashScope / Model Studio API (OpenAI-compatible)** — https://www.alibabacloud.com/help/en/model-studio/
- **Qwen-Agent (tool use / agents framework)** — https://github.com/QwenLM/Qwen-Agent
- **Qwen2.5 technical report** — https://arxiv.org/abs/2412.15115
- **Qwen3 technical report** — https://arxiv.org/abs/2505.09388

**Related notebooks:** `mixture-of-experts` (the MoE machinery behind 30B-A3B / 235B-A22B); `deepseek` (the other top open Chinese family — and a consumer of Qwen distill backbones); `mistral`, `llama` (other open-weight models you'd self-host); `anthropic-claude-api`, `perplexity` (closed frontier / answer-engine alternatives).